# app16-4 — da caixa de fumaça ao modelo (professor)

Lê as coletas que o Node-RED gravou no InfluxDB Cloud (measurement `fumaca`), treina uma
**árvore de decisão** para **fire / not fire** e testa num dia que o modelo nunca viu.

- **fire:** fumaça ou chama. **not fire:** ar, vapor, álcool.
- **Treino:** o dia 11/11 (as coletas das 18:48 e das 22:07, que têm fumaça e chama).
- **Teste:** outro dia, só com condições normais e álcool (vapor, se der). A matriz de confusão
  mostra o alarme falso.
- **No fim:** a última leitura do banco passa no modelo, e o dataset sai em CSV para os alunos
  (`dataset_detector_fumaca_fiap.csv`).

A consulta é em **SQL**, como no Grafana.

> **Execute a primeira célula de código duas vezes.** Na primeira, ela instala as versões fixas
> e reinicia a sessão do Colab; na segunda, só confere.

In [ ]:
# EXECUTE ESTA CELULA DUAS VEZES: na 1a o Colab reinicia a sessao.
#
# Versoes fixas, as mesmas do app17-7: o .pkl gerado aqui e carregado depois (nuvem e
# borda) com EXATAMENTE estas versoes de numpy e scikit-learn.
!pip install -q influxdb3-python pyarrow matplotlib "numpy==2.1.3" "pandas==2.2.3" "scikit-learn==1.6.1" "joblib==1.5.3"

# Reinicia a sessao se o numpy da memoria nao for o recem-instalado
import numpy, os
from importlib.metadata import version
if numpy.__version__ != version("numpy"):
    print("Reiniciando a sessao: rode esta celula de novo.")
    os.kill(os.getpid(), 9)
print("numpy", numpy.__version__, "| ok")

## 1) Conectar no InfluxDB Cloud

Os mesmos valores do nó InfluxDB do Node-RED. O `bucket` do Node-RED é o `database` aqui.

**O token:** guarde nos **Secrets** do Colab (a chave na barra da esquerda), com o nome
`INFLUX_TOKEN`, e libere o acesso deste notebook. Se não houver segredo, a célula usa o que
estiver escrito em `INFLUX_TOKEN`.

A tabela no fim lista as coletas que já estão no banco. O Node-RED marca a tag `coleta` com o
dia (`2026-11-19`); as duas de 11/11 sobem com o dia e a hora do início (célula 1b).

In [ ]:
from influxdb_client_3 import InfluxDBClient3
import pandas as pd

INFLUX_URL    = "https://us-east-1-1.aws.cloud2.influxdata.com"
INFLUX_TOKEN  = "SEU_TOKEN_INFLUX_CLOUD"   # ou o segredo INFLUX_TOKEN do Colab
INFLUX_BUCKET = "IoTSensores"              # o "database" no InfluxDB 3
MEASUREMENT   = "fumaca"

try:
    from google.colab import userdata
    INFLUX_TOKEN = userdata.get("INFLUX_TOKEN")
except Exception:
    pass   # sem o segredo: fica o token escrito acima

client = InfluxDBClient3(host=INFLUX_URL, token=INFLUX_TOKEN, database=INFLUX_BUCKET)

# As coletas que ja estao no banco
coletas = client.query(f'''
SELECT coleta, count(*) AS linhas, min(time) AS inicio, max(time) AS fim
FROM "{MEASUREMENT}"
WHERE time >= now() - INTERVAL '30 days'
GROUP BY coleta
ORDER BY inicio
''', language="sql").to_pandas()
display(coletas)

### 1b) (Uma vez) Subir as duas coletas de 11/11

As coletas de 11/11/2026 foram gravadas pelo Serial (`coleta.py`), antes do MQTT. Esta célula
sobe as duas para o banco. O horário de cada linha é reconstruído: o início da coleta mais a
`amostra` (1 por segundo).

- **22:07**, o coletor do dataset (app16-3): `coleta = 2026-11-11 22:07`.
- **18:48**, o teste dos fenômenos (app16-2), convertida para o formato do dataset:
  `coleta = 2026-11-11 18:48`. Tem só as 7 colunas que os dois programas medem igual.

Precisa de um token com permissão de **escrita**. Mude para `True`, rode **uma vez** e volte
para `False`: rodar de novo grava as mesmas linhas por cima (mesmo horário, sem duplicar).

In [ ]:
SUBIR_COLETAS_11_11 = False   # True uma vez, depois False

RAIZ = "https://raw.githubusercontent.com/norisjunior/FIAP-IoT/fiapiot/eval/app16-Fumaca-SensorFusion/"
SUBIR = [
    ("app16-2-TesteFenomenos/coleta_professor/coleta-20261111-1848-dataset.csv", "2026-11-11 18:48"),
    ("app16-3-Fusao1Hz/coleta_professor/coleta-20261111-2207.csv", "2026-11-11 22:07"),
]

if SUBIR_COLETAS_11_11:
    for caminho, coleta in SUBIR:
        try:
            csv = pd.read_csv(RAIZ + caminho)
        except Exception:
            from google.colab import files
            print("Nao achei o CSV no GitHub: escolha o", caminho.split("/")[-1])
            csv = pd.read_csv(list(files.upload())[0])

        inicio = pd.Timestamp(coleta, tz="America/Sao_Paulo")
        csv["time"] = inicio + pd.to_timedelta(csv["amostra"] - csv["amostra"].min(), unit="s")
        csv["coleta"] = coleta
        csv["device"] = "caixa-fumaca"
        campos = []
        for c in csv.columns:
            if c not in ("time", "coleta", "device"):
                campos.append(c)
        csv[campos] = csv[campos].astype(float)   # como o Node-RED: todo campo decimal

        client.write(record=csv[["time", "coleta", "device"] + campos],
                     data_frame_measurement_name=MEASUREMENT,
                     data_frame_tag_columns=["coleta", "device"],
                     data_frame_timestamp_column="time")
        print(f"{len(csv)} linhas gravadas na coleta {coleta}.")
    print("Rode a celula 1 de novo para conferir.")

## 2) Ler as coletas

Uma linha por segundo. `coleta` e `device` são **tags** (texto); o resto são **fields**
(número). O `estimulo` é o que o botão BOOT marcou: 0 ar, 1 álcool, 2 vapor, 3 fumaça, 4 chama.

In [ ]:
NOMES = {0: "ar", 1: "álcool", 2: "vapor", 3: "fumaça", 4: "chama"}
CORES = {"ar": "#22c55e", "álcool": "#eab308", "vapor": "#a855f7", "fumaça": "#ef4444", "chama": "#3b82f6"}

df = client.query(f'''
SELECT *
FROM "{MEASUREMENT}"
WHERE time >= now() - INTERVAL '30 days'
ORDER BY time
''', language="sql").to_pandas()

df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_convert("America/Sao_Paulo")
df = df.dropna(subset=["estimulo"]).reset_index(drop=True)
df["estimulo"] = df["estimulo"].astype(int)
df["rodada"] = df["rodada"].astype(int)
df["dia"] = df["coleta"].str[:10]

print(len(df), "linhas")
print("Segundos de cada estimulo, por coleta:")
display(pd.crosstab(df["coleta"], df["estimulo"].map(NOMES)))

## 3) Uma coleta no tempo

As faixas coloridas são o botão BOOT, na cor do LED. A curva começa a mudar 15 a 30 s
**depois** do aperto, e a fumaça ainda aparece no começo do ar seguinte.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

COLETA_VER = df["coleta"].iloc[-1]   # a mais recente; ou escreva, por exemplo, "2026-11-11 22:07"
c = df[df["coleta"] == COLETA_VER]

PAINEIS = [("nc0_5", "partícula pequena, nc0_5 (por cm³)"),
           ("raw_ethanol", "gás etanol, raw_ethanol (valor cru: cai quando o gás sobe)"),
           ("temp", "calor (°C)"), ("umid", "umidade (%)")]
trecho = (c["estimulo"] != c["estimulo"].shift()).cumsum()   # segundos seguidos com o mesmo estimulo

fig, axs = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
for ax, (col, rotulo) in zip(axs, PAINEIS):
    for _, t in c.groupby(trecho):
        ax.axvspan(t["time"].iloc[0], t["time"].iloc[-1], color=CORES[NOMES[t["estimulo"].iloc[0]]],
                   alpha=0.15, lw=0)
    ax.plot(c["time"], c[col], color="black", lw=1)
    ax.set_title(rotulo, loc="left", fontsize=10)

legenda = []
for nome, cor in CORES.items():
    legenda.append(Patch(color=cor, alpha=0.4, label=nome))
fig.legend(handles=legenda, loc="upper right", ncol=5)
fig.suptitle(f"coleta {COLETA_VER}", x=0.01, ha="left")
plt.tight_layout()
plt.show()

## 4) Preparar o dataset

- **Sai a rodada 1:** é o aquecimento da caixa.
- **Saem os primeiros segundos de cada trecho:** o rótulo é o aperto do botão, mas a curva muda
  15 a 30 s depois. Nos estímulos saem 30 s; no ar saem 120 s (a fumaça demora a sair).
- **Ficam 7 features:** `temp`, `umid`, `raw_h2`, `raw_ethanol`, `pm2_5`, `pm10`, `nc0_5`. São as
  que os dois programas da caixa medem do mesmo jeito. Saem `amostra`, `rodada`, `pressao` e o
  horário (mudam com o momento, não com o fogo), e `tvoc` e `eco2` (a coleta das 18:48 os mediu
  sem a correção de umidade).
- **O alvo:** `incendio` = `fire` (fumaça ou chama) ou `not fire` (ar, vapor, álcool).

**No dia de teste:** para chegar ao álcool, o BOOT passa por chama e fumaça. Passe em menos de
30 s: o preparo descarta esses segundos, e nenhuma linha sai com o rótulo `fire` errado.

In [ ]:
FEATURES = ["temp", "umid", "raw_h2", "raw_ethanol", "pm2_5", "pm10", "nc0_5"]

def prepara(d):
    d = d.sort_values(["coleta", "time"]).copy()
    # trecho = segundos seguidos com a mesma coleta, rodada e estimulo
    mudou = ((d["coleta"] != d["coleta"].shift()) | (d["rodada"] != d["rodada"].shift())
             | (d["estimulo"] != d["estimulo"].shift()))
    d["seg"] = d.groupby(mudou.cumsum()).cumcount()          # segundos desde o aperto
    descarta = d["estimulo"].map(lambda e: 120 if e == 0 else 30)
    d = d[(d["rodada"] > 1) & (d["seg"] >= descarta)].copy()
    d = d.dropna(subset=FEATURES)
    d["nome"] = d["estimulo"].map(NOMES)
    d["incendio"] = d["estimulo"].map(lambda e: "fire" if e in (3, 4) else "not fire")
    return d

dados = prepara(df)
print(len(dados), "linhas no dataset")
display(pd.crosstab([dados["coleta"], dados["nome"]], dados["incendio"]))

## 5) A fusão no gráfico: cada linha é um ponto

Com dois sensores juntos, cada estímulo ocupa o seu canto: é o **sensor fusion**. Um eixo só
mistura as cores.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for nome, cor in CORES.items():
    g = dados[dados["nome"] == nome]
    axs[0].scatter(g["nc0_5"], g["umid"], s=8, color=cor, alpha=0.6, label=nome)
    axs[1].scatter(g["raw_ethanol"], g["temp"], s=8, color=cor, alpha=0.6, label=nome)
axs[0].set_xscale("log")
axs[0].set_xlabel("partícula pequena, nc0_5 (por cm³, escala log)")
axs[0].set_ylabel("umidade (%)")
axs[1].set_xlabel("gás etanol, raw_ethanol (valor cru: cai quando o gás sobe)")
axs[1].set_ylabel("calor (°C)")
axs[0].legend()
plt.tight_layout()
plt.show()

## 6) Um dia treina, o outro testa

O teste que vale é **outro dia**: a sala muda, e o sensor de gás muda a linha de base de um dia
para o outro (*data drift*). `DIA_TESTE = None` pega o dia mais novo do banco.

Se o banco ainda só tem 11/11, a célula treina na coleta das 18:48 e testa na das 22:07.

In [ ]:
DIA_TREINO = "2026-11-11"
DIA_TESTE = None   # None = o dia mais novo do banco; ou escreva, por exemplo, "2026-11-19"

dias = sorted(dados["dia"].unique())
if DIA_TESTE is None:
    DIA_TESTE = dias[-1]

if DIA_TESTE == DIA_TREINO:
    coletas_dia = sorted(dados.loc[dados["dia"] == DIA_TREINO, "coleta"].unique())
    print("So ha o dia", DIA_TREINO, "no banco: o teste fica com a ultima coleta dele.")
    treino = dados[dados["coleta"].isin(coletas_dia[:-1])]
    teste = dados[dados["coleta"] == coletas_dia[-1]]
else:
    treino = dados[dados["dia"] == DIA_TREINO]
    teste = dados[dados["dia"] == DIA_TESTE]

print("treino:", sorted(treino["coleta"].unique()), f"({len(treino)} linhas)")
print("teste: ", sorted(teste["coleta"].unique()), f"({len(teste)} linhas)")

## 7) O modelo: uma árvore de decisão

Três níveis de perguntas. Dá para ler a regra e ver quais sensores ela combinou.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

arvore = DecisionTreeClassifier(max_depth=3, random_state=42)
arvore.fit(treino[FEATURES], treino["incendio"])

plt.figure(figsize=(16, 7))
plot_tree(arvore, feature_names=FEATURES, class_names=list(arvore.classes_), filled=True,
          impurity=False, fontsize=9)
plt.show()

## 8) O teste: a matriz de confusão

- **Alarme falso:** era `not fire`, o modelo disse `fire`.
- **Fogo perdido:** era `fire`, o modelo disse `not fire`. No dia de teste não há fogo: só dá
  para medir o alarme falso.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

pred = arvore.predict(teste[FEATURES])

matriz = ConfusionMatrixDisplay.from_predictions(teste["incendio"], pred, labels=["not fire", "fire"],
                                                 cmap="Blues", colorbar=False)
matriz.ax_.set_xlabel("o modelo disse")
matriz.ax_.set_ylabel("a verdade (o botão)")
plt.title("teste: " + ", ".join(sorted(teste["coleta"].unique())))
plt.show()

sem_fogo = (teste["incendio"] == "not fire").values
com_fogo = (teste["incendio"] == "fire").values
print(f"Alarme falso: {(pred[sem_fogo] == 'fire').mean():.0%} dos segundos sem fogo")
if com_fogo.any():
    print(f"Fogo perdido: {(pred[com_fogo] == 'not fire').mean():.0%} dos segundos com fogo")
else:
    print("Neste teste nao ha fogo: so da para medir o alarme falso.")

print("\nDe cada estimulo, o que o modelo disse:")
display(pd.crosstab(teste["nome"], pd.Series(pred, index=teste.index, name="o modelo disse"),
                    normalize="index").round(2))

### E se o split fosse aleatório?

A mesma árvore, treinada com o split "de livro": 70% das linhas sorteadas, de todos os dias.
Um segundo e o seguinte são quase iguais, então o teste vira prova com gabarito.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

treino_a, teste_a = train_test_split(dados, test_size=0.3, random_state=42)
arvore_a = DecisionTreeClassifier(max_depth=3, random_state=42)
arvore_a.fit(treino_a[FEATURES], treino_a["incendio"])

aleatorio = accuracy_score(teste_a["incendio"], arvore_a.predict(teste_a[FEATURES]))
outro_dia = accuracy_score(teste["incendio"], pred)
print(f"acerto | split aleatorio: {aleatorio:.2f} | dia de teste: {outro_dia:.2f}")

O número que vale é o do **dia de teste**: é o que o modelo vai encontrar no dia seguinte.

## 9) Salvar o modelo

O `.pkl` é o modelo que a nuvem e a borda usam depois. A entrada é uma linha com as 7 features,
**nesta ordem**.

In [ ]:
import joblib

ARQUIVO_PKL = "modelo_fumaca_incendio.pkl"
joblib.dump(arvore, ARQUIVO_PKL)
print("features, nesta ordem:", FEATURES)

from importlib.metadata import version
for pacote in ("numpy", "pandas", "scikit-learn", "joblib"):
    print(f"{pacote}=={version(pacote)}")

try:
    from google.colab import files
    files.download(ARQUIVO_PKL)
except Exception:
    print(f"{ARQUIVO_PKL} gerado na pasta atual.")

## 10) A última leitura: é incêndio?

Com a caixa publicando, a consulta pega **só a última linha** que chegou ao banco, monta um
DataFrame com as 7 features e passa no modelo. Rode de novo a cada estímulo.

In [ ]:
COLUNAS_SQL = '"' + '", "'.join(FEATURES) + '"'

ultima = client.query(f'''
SELECT time, estimulo, {COLUNAS_SQL}
FROM "{MEASUREMENT}"
WHERE time >= now() - INTERVAL '10 minutes'
ORDER BY time DESC
LIMIT 1
''', language="sql").to_pandas()

if ultima.empty:
    print("Nada nos ultimos 10 min: a caixa esta publicando?")
else:
    hora = pd.to_datetime(ultima["time"].iloc[0], utc=True).tz_convert("America/Sao_Paulo")
    display(ultima[FEATURES])
    resposta = arvore.predict(ultima[FEATURES])[0]
    print(f"{hora:%H:%M:%S} | E incendio? O modelo diz: {resposta}"
          f" | o botao marca: {NOMES[int(ultima['estimulo'].iloc[0])]}")

### (Opcional) Ao vivo: as leituras que estão chegando

A cada 5 s, lê os últimos 30 s do banco e mostra o que o modelo diz. Para parar, use o botão ■
da célula. Nos primeiros 15 a 30 s de cada estímulo, o modelo e o botão discordam: a curva
ainda não mudou.

In [ ]:
import time

try:
    for _ in range(60):   # uns 5 minutos
        agora = client.query(f'''
        SELECT time, estimulo, {COLUNAS_SQL}
        FROM "{MEASUREMENT}"
        WHERE time >= now() - INTERVAL '30 seconds'
        ORDER BY time
        ''', language="sql").to_pandas().dropna(subset=FEATURES)

        if agora.empty:
            print("Nada nos ultimos 30 s: a caixa esta publicando?")
        else:
            pred_agora = arvore.predict(agora[FEATURES])
            hora = pd.to_datetime(agora["time"].iloc[-1], utc=True).tz_convert("America/Sao_Paulo")
            maioria = pd.Series(pred_agora[-10:]).mode()[0]
            print(f"{hora:%H:%M:%S} | modelo agora: {pred_agora[-1]:8s}"
                  f" | ultimos 10 s: {maioria:8s}"
                  f" | o botao marca: {NOMES[int(agora['estimulo'].iloc[-1])]}")
        time.sleep(5)
except KeyboardInterrupt:
    print("parado")

## 11) Exportar o dataset para os alunos

Gera o `dataset_detector_fumaca_fiap.csv` com o dataset já preparado (seção 4): `coleta`,
`estimulo`, as 7 features e `incendio`. É o arquivo que o aluno abre no `caixa_de_fumaca.ipynb`.
Com o dia de teste no banco, o notebook do aluno treina em 11/11 e testa nele.

In [ ]:
ARQUIVO_CSV = "dataset_detector_fumaca_fiap.csv"

saida = pd.DataFrame()
saida["coleta"] = dados["coleta"].values
saida["estimulo"] = dados["nome"].values
for f in FEATURES:
    if f in ("raw_h2", "raw_ethanol"):
        saida[f] = dados[f].round().astype(int).values
    else:
        saida[f] = dados[f].round(2).values
saida["incendio"] = dados["incendio"].values

saida.to_csv(ARQUIVO_CSV, index=False, encoding="utf-8")
print(f"{ARQUIVO_CSV}: {len(saida)} linhas")
display(pd.crosstab(saida["coleta"], saida["incendio"]))

try:
    from google.colab import files
    files.download(ARQUIVO_CSV)
except Exception:
    print(f"{ARQUIVO_CSV} gerado na pasta atual.")